In [1]:
import pandas as pd
import dotenv
import os
import amaru.config.config as config
dotenv.load_dotenv()

path_config = os.getenv("BASE_PATH_DATA_CLEANED")

In [2]:
import pathlib
def union_all_files_whitout_label():
    list_files = list(pathlib.Path(path_config).rglob("*.csv"))
    df_union = pd.concat([pd.read_csv(file).drop(columns=["label"]) for file in list_files])
    return df_union

def union_all_files_with_label():
    list_files = list(pathlib.Path(path_config).rglob("*.csv"))
    df_union = pd.concat([pd.read_csv(file) for file in list_files])

    return df_union

In [3]:
import numpy as np

def get_correlation_matrix_spearman(df):
    return df.corr(method="spearman")


def get_columns_to_drop_by_correlation(df, threshold=0.9):
    corr = get_correlation_matrix_spearman(df)
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]
    return to_drop




In [4]:
df_with_label = union_all_files_with_label()
df_with_label.drop(columns="timestamp", inplace=True)
columns_to_drop_by_correlation = get_columns_to_drop_by_correlation(df_with_label)


In [5]:
from sklearn.feature_selection import VarianceThreshold
def variance_threshold_selector(df, threshold=0.01):
    selector = VarianceThreshold(threshold=threshold)
    selector.fit(df)
    
    columns_to_keep = selector.get_feature_names_out()
    columns_to_drop = df.columns[~selector.get_support()].tolist()
    
    print(f"Features originales:  {df.shape[1]}")
    print(f"Features que se mantienen: {len(columns_to_keep)}")
    print(f"Features eliminadas:  {len(columns_to_drop)}")
    print(f"Columnas eliminadas: {columns_to_drop}")
    
    return columns_to_drop  


df_without_label = union_all_files_whitout_label()
# df_without_label.drop(columns=columns_to_drop_by_correlation, inplace=True)
df_without_label.drop(columns='timestamp', inplace=True)
columns_to_drop_by_variance = variance_threshold_selector(df_without_label)


Features originales:  47
Features que se mantienen: 47
Features eliminadas:  0
Columnas eliminadas: []


In [6]:
print(len(columns_to_drop_by_variance))
print(len(columns_to_drop_by_correlation))
len(list(set(columns_to_drop_by_variance + columns_to_drop_by_correlation)))

0
25


25

In [7]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split
def mutual_info_classif_selector(X, y):
    print(f"Shape de X: {X.shape}")
    print(f"Shape de y: {y.shape}")
    print(f"Columnas disponibles: {X.columns.tolist()}")
    
    if X.shape[1] == 0:
        raise ValueError("X no tiene columnas — revisa el paso de eliminación previo")
    
    mi = mutual_info_classif(X.values, np.array(y))
    mi_scores = pd.Series(mi, index=X.columns)
    mi_scores = mi_scores.sort_values(ascending=False)
    return mi_scores


df_with_label = union_all_files_with_label()

print(f"Columnas totales iniciales: {df_with_label.shape[1]}")
print(f"Columnas a eliminar por correlación: {len(columns_to_drop_by_correlation)}")
print(f"Columnas a eliminar por varianza: {len(columns_to_drop_by_variance)}")



total_columns_to_drop = list(set(columns_to_drop_by_variance + columns_to_drop_by_correlation))

print(f"Columnas que realmente se van a eliminar: {len(total_columns_to_drop)}")

df_with_label = df_with_label.drop(columns=total_columns_to_drop)
df_with_label = df_with_label.drop(columns="timestamp", errors='ignore')

print(f"Columnas restantes: {df_with_label.shape[1]}")
print(f"Columnas restantes (nombres): {df_with_label.columns.tolist()}")

# Separar X e y — mantener como DataFrame
X = df_with_label.drop(columns="label")
y = df_with_label["label"]

print(f"Shape X final: {X.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

mi_scores = mutual_info_classif_selector(X_train, y_train)


Columnas totales iniciales: 49
Columnas a eliminar por correlación: 25
Columnas a eliminar por varianza: 0
Columnas que realmente se van a eliminar: 25
Columnas restantes: 23
Columnas restantes (nombres): ['total_fwd_packets', 'total_backward_packets', 'total_length_of_fwd_packets', 'total_length_of_bwd_packets', 'fwd_packet_length_min', 'fwd_packet_length_std', 'bwd_packet_length_min', 'bwd_packet_length_std', 'flow_bytes-s', 'flow_packets-s', 'flow_iat_mean', 'flow_iat_std', 'flow_iat_min', 'fwd_iat_total', 'fwd_iat_std', 'fwd_iat_min', 'bwd_iat_total', 'down-up_ratio', 'bwd_packets-s', 'active_mean', 'active_std', 'idle_std', 'label']
Shape X final: (2606252, 22)
Shape de X: (2085001, 22)
Shape de y: (2085001,)
Columnas disponibles: ['total_fwd_packets', 'total_backward_packets', 'total_length_of_fwd_packets', 'total_length_of_bwd_packets', 'fwd_packet_length_min', 'fwd_packet_length_std', 'bwd_packet_length_min', 'bwd_packet_length_std', 'flow_bytes-s', 'flow_packets-s', 'flow_iat_

In [11]:
mi_scores > 0.006

down-up_ratio                   True
fwd_packet_length_std           True
bwd_packet_length_std           True
total_length_of_bwd_packets     True
flow_iat_mean                   True
fwd_iat_total                   True
total_length_of_fwd_packets     True
flow_iat_min                    True
flow_packets-s                  True
fwd_iat_min                     True
flow_bytes-s                    True
bwd_packets-s                   True
flow_iat_std                    True
fwd_iat_std                     True
bwd_iat_total                   True
bwd_packet_length_min          False
fwd_packet_length_min          False
total_fwd_packets              False
active_mean                    False
total_backward_packets         False
idle_std                       False
active_std                     False
dtype: bool